This notebook is the best representation of a neural network without abstracting to the level of a single neuron as in the notebook backpropagation.ipynb

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score as r2, mean_squared_error as mse, accuracy_score as acc, precision_score as prec

Testing out the logic before implementing a class for performing experiments.

In [2]:
X = np.random.rand(5, 3)
print("X: ", X.shape, "\n")

W1 = np.random.rand(3, 2)
print("W1: ", W1.shape, "\n")

W2 = np.random.rand(2, 2)
print("W2: ", W2.shape, "\n")

W3 = np.random.rand(2, 1)
print("W3: ",W1.shape, "\n")

original = np.tanh(np.random.rand(3, 1))
print("Original: ",original.shape, "\n")

y = (X.dot(original)>0.5).astype(int)
print("y: ",y.shape, "\n")

X:  (5, 3) 

W1:  (3, 2) 

W2:  (2, 2) 

W3:  (3, 2) 

Original:  (3, 1) 

y:  (5, 1) 



In [3]:
z1 = X.dot(W1)
a1 = np.tanh(z1)
print("z1: ", z1.shape)
print("a1: ", a1.shape)

z2 = a1.dot(W2)
a2 = np.tanh(z2)
print("z2: ", z2.shape)
print("a2: ", a2.shape)

z3 = a2.dot(W3)
a3 = 1/ (1 + np.exp(-z3))
print("z3: ", z3.shape)
print("a3: ", a3.shape)

z1:  (5, 2)
a1:  (5, 2)
z2:  (5, 2)
a2:  (5, 2)
z3:  (5, 1)
a3:  (5, 1)


In [4]:
dz3 = (a3 - y) * (a3 * (1 - a3))
print(dz3.shape)

dw3 = a2.T@dz3
print(dw3.shape)

dz2 = (dz3@W3.T) * (1 - a2**2)
print(dz2.shape)

dw2 = a1.T@dz2
print(dw2.shape)

dz1 = (dz2@W2.T) * (1 - a1**2)
print(dz1.shape)

dw1 = X.T@dz1
print(dw1.shape)

(5, 1)
(2, 1)
(5, 2)
(2, 2)
(5, 2)
(3, 2)


Class level implementation of the above model

In [40]:
class ClassificationNeuralNetwork():
    
    def __init__(self, input):
        
        # layer 1 weights & bias
        self.W1 = np.random.rand(input, 3)
        self.b1 = np.random.rand(1, 3)
        
        # layer 2 weights & bias
        self.W2 = np.random.rand(3, 2)
        self.b2 = np.random.rand(1, 2)
        
        # output layer weights & bias
        self.W3 = np.random.rand(2, 1)
        self.b3 = np.random.rand(1, 1)
        
    def forward(self, X):
        
        # layer 1 
        self.z1 = X@self.W1 + self.b1
        self.a1 = np.tanh(self.z1) # tanh as activation
        
        # layer 2
        self.z2 = self.a1@self.W2 + self.b2
        self.a2 = np.tanh(self.z2) # tanh as activation
        
        # layer 3 
        self.z3 = self.a2@self.W3 + self.b3
        self.a3 = 1/ (1 + np.exp(-self.z3))
        
    def backward(self, X, y):
        
        # layer 3
        self.dz3 = (self.a3 - y) * (self.a3 * (1- self.a3))
        self.dw3 = self.a2.T@self.dz3
        
        #print(self.dz3.shape)
        # layer 2
        self.dz2 = (self.dz3@self.W3.T) * (1 - self.a2**2)
        self.dw2 = self.a1.T@self.dz2
        
        # layer 1
        self.dz1 = (self.dz2@self.W2.T) * (1 - (self.a1)**2)
        self.dw1 = X.T@self.dz1
        
        
    def update(self, lr):
        
        # update weights
        self.W1 -= self.dw1 * lr
        self.W2 -= self.dw2 * lr 
        self.W3 -= self.dw3 * lr
        
        # update biases
        self.b1 -= np.sum(self.dz1, axis=0).reshape(1, -1) * lr
        self.b2 -= np.sum(self.dz2, axis=0).reshape(1, -1) * lr
        self.b3 -= np.sum(self.dz3, axis=0).reshape(1, -1) * lr

    def fit(self, X, y, n_epochs, lr):
        
        for epoch in range(n_epochs):
            self.forward(X)
            self.backward(X, y)
            self.update(lr)
            
            if (epoch + 1)%10 == 0:
                print(f"(Epoch {epoch+1})BCE: ", np.round(np.mean((-(y*np.log(self.a3) + (1 - y)*np.log(1 - self.a3)))), 2)) #−(ylog(z)+(1−y)log(1−z))
        
    def predict(self, X, threshold = 0.5):
        self.forward(X)
        #return (self.a3 >= threshold).astype(int)
        return self.a3

In [25]:
class RegressionNeuralNetwork():
    
    def __init__(self, input):
        
        # layer 1 weights & bias
        self.W1 = np.random.rand(input, 3)
        self.b1 = np.random.rand(1, 3)
        
        # layer 2 weights & bias
        self.W2 = np.random.rand(3, 2)
        self.b2 = np.random.rand(1, 2)
        
        # output layer weights & bias
        self.W3 = np.random.rand(2, 1)
        self.b3 = np.random.rand(1, 1)
        
    def forward(self, X):
        
        # layer 1 
        self.z1 = X@self.W1 + self.b1
        self.a1 = np.tanh(self.z1) # tanh as activation
        
        # layer 2
        self.z2 = self.a1@self.W2 + self.b2
        self.a2 = np.tanh(self.z2) # tanh as activation
        
        # layer 3 
        self.z3 = self.a2@self.W3 + self.b3
        self.a3 = self.z3
        
    def backward(self, X, y):
        
        #print((self.a3 - y).shape)
        # layer 3
        self.dz3 = (self.a3 - y) * (np.ones_like(self.a3))
        #print(self.dz3.shape)
        self.dw3 = self.a2.T@self.dz3
        #print(self.dw3.shape)
        
        # layer 2
        self.dz2 = (self.dz3@self.W3.T) * (1 - (self.a2)**2)
        #print(self.dz2.shape)
        self.dw2 = self.a1.T@self.dz2
        #print(self.dw2.shape)
        
        # layer 1
        self.dz1 = (self.dz2@self.W2.T) * (1 - (self.a1)**2)
        #print(self.dz1.shape)
        self.dw1 = X.T@self.dz1
        #print(self.dw1.shape)
        
    def update(self, lr):
        
        # update weights
        self.W1 -= self.dw1 * lr
        self.W2 -= self.dw2 * lr 
        self.W3 -= self.dw3 * lr
        #print(self.W1.shape)
        #print(self.W2.shape)
        #print(self.W3.shape)
        
        # update biases
        self.b1 -= np.sum(self.dz1, axis=0).reshape(1, -1) * lr
        self.b2 -= np.sum(self.dz2, axis=0).reshape(1, -1) * lr
        self.b3 -= np.sum(self.dz3, axis=0).reshape(1, -1) * lr
        #print(self.b1.shape)
        #print(self.b2.shape)
        #print(self.b3.shape)

    def fit(self, X, y, n_epochs, lr):
        
        for epoch in range(n_epochs):
            self.forward(X)
            self.backward(X, y)
            self.update(lr)
            
            #if (epoch + 1)%10 == 0: print()
                #print(f"(Epoch {epoch+1})BCE: ", np.round(np.mean((self.a3 - y)**2), 2)) #mse
        
    def predict(self, X):
        self.forward(X)
        return self.a3

Testing on synthetic data

In [7]:
# Synthetic data for regression and classification

def synthetic_regression_data(N, # number of datapoints to generate
                              n_input, # number of input variables 
                              ): 
    
    # 1. determine weights and bias randomly for the data
    W = np.random.randn(n_input)
    b = np.random.randn()
    
    # 2. Generate data
    X = np.random.randint(10, size=(N, n_input))
    y = X.dot(W) + b + np.random.rand(N)
    
    
    # 4. return the W, b and data
    return X, y
    
    
def synthetic_classification_data(N, n_input):
    
    # 1. determine weights and bias randomly for the data
    W = np.random.randn(n_input)
    b = np.random.randn()
    
    # 2. Generate data
    X = np.random.randint(10, size=(N, n_input))
    y_ = X.dot(W) + b + np.random.rand()
    threshold = np.random.random()
    y = (y_ >= threshold).astype(int)
    
    
    # 4. return the W, b and data
    return threshold, X, y
    

In [39]:
X_reg, y_reg = synthetic_regression_data(100, 5)

reg_model = RegressionNeuralNetwork(5)

reg_model.fit(X_reg, y_reg.reshape(-1, 1), 155, 0.001)
print("Fit complete")

y_pred_reg = reg_model.predict(X_reg)

print("MSE: ", np.mean((y_pred_reg - y_reg)**2))

Fit complete
MSE:  11.40564842187587


In [47]:
cls_threshold, X_cls, y_cls = synthetic_classification_data(100, 5)

cls_model = ClassificationNeuralNetwork(5)

cls_model.fit(X_cls, y_cls.reshape(-1, 1), 150, 0.1)
print("Fit complete")

y_pred_cls = cls_model.predict(X_cls, cls_threshold)

print("BCE: ", np.round(np.mean((-(y_cls.reshape(-1, 1)*np.log(y_pred_cls) + (1 - y_cls.reshape(-1, 1))*np.log(1 - y_pred_cls)))), 2)) #−(ylog(z)+(1−y)log(1−z))

(Epoch 10)BCE:  0.04
(Epoch 20)BCE:  0.03
(Epoch 30)BCE:  0.02
(Epoch 40)BCE:  0.02
(Epoch 50)BCE:  0.02
(Epoch 60)BCE:  0.02
(Epoch 70)BCE:  0.02
(Epoch 80)BCE:  0.02
(Epoch 90)BCE:  0.01
(Epoch 100)BCE:  0.01
(Epoch 110)BCE:  0.01
(Epoch 120)BCE:  0.01
(Epoch 130)BCE:  0.01
(Epoch 140)BCE:  0.01
(Epoch 150)BCE:  0.01
Fit complete
BCE:  0.01


Testing on real datasets

In [10]:
reg_data = pd.read_csv('./datasets/CASP.csv')
reg_data.info()

X_casp, y_casp = reg_data.drop(['RMSD'], axis=1), reg_data['RMSD']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45730 entries, 0 to 45729
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RMSD    45730 non-null  float64
 1   F1      45730 non-null  float64
 2   F2      45730 non-null  float64
 3   F3      45730 non-null  float64
 4   F4      45730 non-null  float64
 5   F5      45730 non-null  float64
 6   F6      45730 non-null  float64
 7   F7      45730 non-null  float64
 8   F8      45730 non-null  int64  
 9   F9      45730 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 3.5 MB


In [21]:
reg_model_real = RegressionNeuralNetwork(X_casp.shape[-1])
reg_model_real.fit(np.asarray(X_casp), np.asarray(y_casp).reshape(-1, 1), 100, 0.00001)
y_reg_model_real = reg_model_real.predict(X_casp)

print("R2 score : ", r2(np.asarray(y_casp).reshape(-1, 1), y_reg_model_real))
print("MSE : ", mse(np.asarray(y_casp).reshape(-1, 1), y_reg_model_real))


(Epoch 10)BCE:  37.43
(Epoch 20)BCE:  37.42
(Epoch 30)BCE:  37.41
(Epoch 40)BCE:  37.4
(Epoch 50)BCE:  37.4
(Epoch 60)BCE:  37.39
(Epoch 70)BCE:  37.38
(Epoch 80)BCE:  37.37
(Epoch 90)BCE:  37.36
(Epoch 100)BCE:  37.36
R2 score :  0.002047587117616878
MSE :  37.3562720129


In [23]:
from ucimlrepo import fetch_ucirepo 
  
# fetching classification dataset 
toxicity = fetch_ucirepo(id=728) 
  
# data (as pandas dataframes) 
X_toxic = toxicity.data.features 
y_toxic = toxicity.data.targets 

# for simplicity sake we are skipping all the categorical features and considering only around 10 random features

toxicity_features = ['MATS3v', 'MATS3s', 'MATS3p', 'nHBDon_Lipinski', 'minHBint8', 'MATS3e', 'MATS3c', 'MATS3m'] 
X_toxic = X_toxic[toxicity_features]

X_toxic = np.array(X_toxic)
y_toxic = (np.array(y_toxic) != "NonToxic").astype(int).reshape(-1, 1)

In [26]:
cls_model_real = ClassificationNeuralNetwork(X_toxic.shape[-1])
cls_model_real.fit(X_toxic, y_toxic, 100, 0.01)
y_cls_model_real = cls_model_real.predict(X_toxic)

print("Accuracy score : ", acc(y_toxic, (y_cls_model_real >= 0.5).astype(int).reshape(-1, 1)))
print("Precision score : ", prec(y_toxic, (y_cls_model_real >= 0.5).astype(int).reshape(-1, 1)))


(Epoch 10)BCE:  0.75
(Epoch 20)BCE:  0.64
(Epoch 30)BCE:  0.63
(Epoch 40)BCE:  0.62
(Epoch 50)BCE:  0.61
(Epoch 60)BCE:  0.6
(Epoch 70)BCE:  0.6
(Epoch 80)BCE:  0.59
(Epoch 90)BCE:  0.58
(Epoch 100)BCE:  0.57
Accuracy score :  0.7017543859649122
Precision score :  1.0


# Major Learnings

1. Clear understanding of matrix operations
2. Flow of gradients in backpropagation